# Face Mask Detection - Model Training (3-Class)

Dataset: vijaykumar1799/face-mask-detection (Kaggle)

Classes:
- 0: with_mask              (correctly worn mask)
- 1: mask_weared_incorrect  (mask on chin / neck / below nose)
- 2: without_mask           (no mask at all)

Pipeline:
- Part A: Quick benchmark - YOLOv8n-cls vs MobileNetV2 vs EfficientNetB0 (8 epochs each)
- Part B: Full optimized training of the best model (EfficientNetB0)
- Part C: TFLite export with FP16 and INT8 quantization

Expected results:
- Validation accuracy: >= 95% (3-class is harder than 2-class)
- TFLite FP16 model size: ~5 MB (down from ~21 MB Keras model)

## Section 1 - Setup

In [1]:
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')   # non-interactive backend required on Kaggle
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Allow GPU memory to grow incrementally instead of allocating all at once
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow version :', tf.__version__)
print('Keras version      :', keras.__version__)
print('GPUs available     :', len(tf.config.list_physical_devices('GPU')))
print('Python version     :', sys.version)

2026-05-12 13:59:20.525708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778594360.767224      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778594360.835457      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778594361.361897      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778594361.361935      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778594361.361938      57 computation_placer.cc:177] computation placer alr

TensorFlow version : 2.19.0
Keras version      : 3.10.0
GPUs available     : 2
Python version     : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


## Section 2 - Configuration

In [2]:
# --- Paths ---
# Dataset is mounted automatically in Kaggle at this path
DATA_DIR = '/kaggle/input/datasets/vijaykumar1799/face-mask-detection/Dataset'

# All outputs (models, plots, metrics) go here
OUT_DIR  = '/kaggle/working/models'
os.makedirs(OUT_DIR, exist_ok=True)

# --- Dataset ---
# Subfolder names inside DATA_DIR — must match the actual folder names exactly
MASK_FOLDER      = 'with_mask'             # correctly worn mask
INCORRECT_FOLDER = 'mask_weared_incorrect'  # worn on chin / below nose / neck
NOMASK_FOLDER    = 'without_mask'          # no mask at all

# Class list — the position in this list determines the integer label (0, 1, 2)
CLASS_FOLDERS = [MASK_FOLDER, INCORRECT_FOLDER, NOMASK_FOLDER]
CLASS_LABELS  = ['with_mask', 'mask_weared_incorrect', 'without_mask']
NUM_CLASSES   = len(CLASS_LABELS)   # 3

# Colors used in plots — one per class
CLASS_COLORS = ['#2ECC71', '#F39C12', '#E74C3C']   # green, orange, red

# --- Image ---
IMG_SIZE   = 224    # EfficientNetB0 and MobileNetV2 both use 224x224
TEST_SPLIT = 0.20   # 80% train, 20% validation

# --- Training ---
BATCH_SIZE      = 32
QUICK_EPOCHS    = 8     # used for the comparison benchmark only
PHASE1_EPOCHS   = 50    # full training phase 1: feature extraction
PHASE2_EPOCHS   = 30    # full training phase 2: fine-tuning
UNFREEZE_LAYERS = 15  # reduced from 30 to prevent catastrophic forgetting    # number of top backbone layers to unfreeze in phase 2

# --- Verify all dataset folders exist before proceeding ---
total_images = 0
for folder_name in CLASS_FOLDERS:
    folder_path = os.path.join(DATA_DIR, folder_name)
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(
            f'Expected folder not found: {folder_path}\n'
            f'Check that DATA_DIR is correct and the dataset is attached to this notebook.'
        )
    count = len([f for f in os.listdir(folder_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    total_images += count
    print(f'{folder_name}: {count} images')

print(f'\nTotal images     : {total_images}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Output directory : {OUT_DIR}')

with_mask: 2994 images
mask_weared_incorrect: 2994 images
without_mask: 2994 images

Total images     : 8982
Number of classes: 3
Output directory : /kaggle/working/models


## Section 3 - Load Dataset

In [3]:
def load_dataset(data_dir, img_size):
    """
    Read all images from the three class subfolders.

    Returns:
        X : numpy array of shape (N, img_size, img_size, 3), float32, range [0, 1]
        y : numpy array of shape (N,), int32
              0 = with_mask
              1 = mask_weared_incorrect
              2 = without_mask
    """
    X, y = [], []

    for label_idx, folder_name in enumerate(CLASS_FOLDERS):
        folder_path = os.path.join(data_dir, folder_name)
        files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        print(f'Loading {CLASS_LABELS[label_idx]} ({folder_name}): {len(files)} images')

        for fname in files:
            img_path = os.path.join(folder_path, fname)
            img = cv2.imread(img_path)
            if img is None:
                continue   # skip corrupted files silently
            img = cv2.resize(img, (img_size, img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            X.append(img)
            y.append(label_idx)

    X = np.array(X, dtype=np.float32)  # keep [0, 255] — pretrained models preprocess internally
    y = np.array(y, dtype=np.int32)
    return X, y


print('Loading dataset...')
t0 = time.time()
X, y = load_dataset(DATA_DIR, IMG_SIZE)
print(f'Done in {time.time() - t0:.1f}s')
print(f'Total images loaded : {len(X)}')
print(f'Array shape         : {X.shape}')
for i, label in enumerate(CLASS_LABELS):
    print(f'  Class {i} ({label}): {np.sum(y == i)} images')

Loading dataset...
Loading with_mask (with_mask): 2994 images
Loading mask_weared_incorrect (mask_weared_incorrect): 2994 images
Loading without_mask (without_mask): 2994 images
Done in 123.8s
Total images loaded : 8982
Array shape         : (8982, 224, 224, 3)
  Class 0 (with_mask): 2994 images
  Class 1 (mask_weared_incorrect): 2994 images
  Class 2 (without_mask): 2994 images


In [4]:
# --- Train / validation split ---
# stratify=y ensures all 3 classes are proportionally represented in both splits
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=TEST_SPLIT,
    stratify=y,
    random_state=42
)

print(f'Train : {len(X_train)} images')
print(f'Val   : {len(X_val)} images')
print()
for i, label in enumerate(CLASS_LABELS):
    print(f'  Train {label}: {np.sum(y_train == i)}   Val: {np.sum(y_val == i)}')

# --- Class weights to handle imbalanced classes (mask_weared_incorrect has fewer images) ---
from sklearn.utils.class_weight import compute_class_weight
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))
print('Class weights (to balance imbalanced dataset):')
for i, label in enumerate(CLASS_LABELS):
    print(f'  Class {i} ({label}): weight = {class_weight_dict[i]:.3f}')


Train : 7185 images
Val   : 1797 images

  Train with_mask: 2395   Val: 599
  Train mask_weared_incorrect: 2395   Val: 599
  Train without_mask: 2395   Val: 599
Class weights (to balance imbalanced dataset):
  Class 0 (with_mask): weight = 1.000
  Class 1 (mask_weared_incorrect): weight = 1.000
  Class 2 (without_mask): weight = 1.000


## Section 4 - Exploratory Data Analysis

In [5]:
# --- Sample images (one row per class) ---
fig, axes = plt.subplots(NUM_CLASSES, 6, figsize=(16, 3 * NUM_CLASSES))
fig.suptitle('Sample Images - one row per class', fontsize=13)

for row, cls_idx in enumerate(range(NUM_CLASSES)):
    indices = np.where(y == cls_idx)[0][:6]
    for col, idx in enumerate(indices):
        axes[row][col].imshow(X[idx])
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(
                CLASS_LABELS[cls_idx],
                fontsize=9,
                color=CLASS_COLORS[cls_idx]
            )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'sample_images.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [1.0..211.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..186.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [4.0..194.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [4.0..253.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [5.0..255.0].
Clipping input data to the valid range for imshow with RGB dat

Saved: sample_images.png


In [6]:
# --- Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Class Distribution', fontsize=13)

for ax, split_y, title in zip(axes, [y_train, y_val], ['Train', 'Validation']):
    counts = [np.sum(split_y == i) for i in range(NUM_CLASSES)]
    bars = ax.bar(CLASS_LABELS, counts, color=CLASS_COLORS)
    ax.set_title(title)
    ax.set_ylabel('Image count')
    ax.tick_params(axis='x', rotation=15)
    for bar, count in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            str(count),
            ha='center', fontweight='bold', fontsize=10
        )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'class_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

Saved: class_distribution.png


## Section 5 - Data Augmentation

Augmentation is applied inside the model graph, so it only activates during
training (when training=True). During validation and inference it has no effect.

In [7]:
augment_pipeline = keras.Sequential(
    [
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.20),           # +/- 20% rotation
        layers.RandomZoom((-0.15, 0.15)),      # zoom in or out by 15%
        layers.RandomTranslation(0.10, 0.10),  # shift up/down and left/right
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.15),
    ],
    name='augmentation'
)

# --- Preview augmented images ---
sample_img = X_train[:1]   # one image, shape (1, 224, 224, 3)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Augmentation Preview (same source image, different random transforms)', fontsize=12)

axes[0][0].imshow(sample_img[0])
axes[0][0].set_title('Original')
axes[0][0].axis('off')

for i in range(1, 10):
    row, col = divmod(i, 5)
    aug_img = augment_pipeline(sample_img, training=True)[0].numpy()
    axes[row][col].imshow(np.clip(aug_img, 0, 1))
    axes[row][col].set_title(f'Aug {i}')
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'augmentation_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: augmentation_preview.png')

I0000 00:00:1778594512.870544      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778594512.876562      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [7.0..175.0].


Saved: augmentation_preview.png


---
# Part A - Model Comparison (8-epoch Benchmark)

Three architectures are trained for 8 epochs each under identical conditions
to compare accuracy, model size, and training speed on the 3-class problem.

| Architecture     | Backbone            | Pretrained | Notes |
|------------------|---------------------|------------|-------|
| YOLOv8n-cls (TF) | CSPDarknet + C2f    | No         | TF reimplementation of YOLOv8 nano |
| MobileNetV2      | Inverted residuals  | ImageNet   | Fastest inference, smallest size |
| EfficientNetB0   | MBConv + SE blocks  | ImageNet   | Best accuracy/size tradeoff |

## Section 6 - Define Model Architectures

In [8]:
# =============================================================================
# Model 1: YOLOv8n-cls reimplemented in TF/Keras
# =============================================================================
# YOLOv8 classification uses a CSPDarknet backbone with C2f blocks.
# The 'nano' variant has ~2.7M parameters.
# Reproduced here in pure TF so no PyTorch dependency is needed.
# (Official version: pip install ultralytics -> from ultralytics import YOLO)

def conv_bn_silu(x, filters, kernel_size, strides=1):
    """Conv2D -> BatchNorm -> SiLU activation. Core YOLOv8 building block."""
    x = layers.Conv2D(
        filters, kernel_size,
        strides=strides, padding='same', use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)   # SiLU and swish are mathematically equivalent
    return x


def bottleneck_block(x, filters):
    """Residual bottleneck: two conv layers with a skip connection."""
    residual = x
    x = conv_bn_silu(x, filters, 3)
    x = conv_bn_silu(x, filters, 3)
    return layers.Add()([residual, x])


def c2f_block(x, filters, n_bottlenecks=1):
    """
    C2f block (Cross-Stage Partial with 2 paths and feature fusion).
    Splits feature map into two halves, applies bottlenecks on one half,
    then concatenates all intermediate outputs for richer gradient flow.
    """
    x    = conv_bn_silu(x, filters, 1)
    half = filters // 2

    split_a = x[:, :, :, :half]   # first half — passed through directly
    split_b = x[:, :, :, half:]   # second half — goes through bottlenecks

    parts = [split_a, split_b]
    for _ in range(n_bottlenecks):
        split_b = bottleneck_block(split_b, half)
        parts.append(split_b)

    x = layers.Concatenate()(parts)
    x = conv_bn_silu(x, filters, 1)
    return x


def build_yolov8_cls_tf(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    """
    YOLOv8n classification backbone in TF/Keras.
    Input:  (img_size, img_size, 3)
    Output: (num_classes,) softmax probabilities
    """
    inp = keras.Input(shape=(img_size, img_size, 3), name='input')
    x   = augment_pipeline(inp)
    x   = layers.Rescaling(1.0 / 255.0)(x)  # normalize to [0,1] — YOLOv8 has no built-in preprocessing

    # Stem
    x = conv_bn_silu(x,  16, 3, strides=2)   # 224x224 -> 112x112

    # Stage 1
    x = conv_bn_silu(x,  32, 3, strides=2)   # -> 56x56
    x = c2f_block(x,  32, n_bottlenecks=1)

    # Stage 2
    x = conv_bn_silu(x,  64, 3, strides=2)   # -> 28x28
    x = c2f_block(x,  64, n_bottlenecks=2)

    # Stage 3
    x = conv_bn_silu(x, 128, 3, strides=2)   # -> 14x14
    x = c2f_block(x, 128, n_bottlenecks=2)

    # Stage 4
    x = conv_bn_silu(x, 256, 3, strides=2)   # -> 7x7
    x = c2f_block(x, 256, n_bottlenecks=1)

    # Classification head
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inp, out, name='YOLOv8n_cls_TF')


print('YOLOv8n-cls (TF) builder defined')

YOLOv8n-cls (TF) builder defined


In [9]:
# =============================================================================
# Model 2: MobileNetV2 (ImageNet pretrained backbone, frozen)
# =============================================================================
def build_mobilenetv2(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    base = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # frozen for the benchmark

    inp = keras.Input(shape=(img_size, img_size, 3))
    x   = augment_pipeline(inp)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inp, out, name='MobileNetV2')


# =============================================================================
# Model 3: EfficientNetB0 (ImageNet pretrained backbone, frozen)
# =============================================================================
def build_efficientnetb0(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    base = EfficientNetB0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # frozen for the benchmark

    inp = keras.Input(shape=(img_size, img_size, 3))
    x   = augment_pipeline(inp)
    x   = base(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inp, out, name='EfficientNetB0')


print('MobileNetV2 builder defined')
print('EfficientNetB0 builder defined')

MobileNetV2 builder defined
EfficientNetB0 builder defined


## Section 7 - Run Quick Benchmark

In [10]:
def run_quick_benchmark(model, epochs=QUICK_EPOCHS, lr=1e-3):
    """
    Compile and train a model for `epochs` epochs.
    Returns a dict with accuracy, approximate weight size, and training time.

    Note: Keras 3 requires the weight filename to end in '.weights.h5'
    (dot before 'weights', not underscore).
    """
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    callbacks = [
        EarlyStopping(
            monitor='val_accuracy', patience=3,
            restore_best_weights=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-6, verbose=0
        ),
    ]

    start   = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    elapsed = time.time() - start

    y_pred  = np.argmax(model.predict(X_val, verbose=0), axis=1)
    val_acc = accuracy_score(y_val, y_pred)

    # Keras 3 requires filename to end in '.weights.h5' (dot, not underscore)
    weight_path = f'/tmp/{model.name}.weights.h5'
    model.save_weights(weight_path)
    size_mb = os.path.getsize(weight_path) / 1e6
    os.remove(weight_path)

    return {
        'name'        : model.name,
        'params_M'    : round(model.count_params() / 1e6, 2),
        'size_MB'     : round(size_mb, 2),
        'val_acc_pct' : round(val_acc * 100, 2),
        'best_val_acc': round(max(history.history['val_accuracy']) * 100, 2),
        'time_s'      : round(elapsed, 1),
        'history'     : history.history,
    }


benchmark_results = []

model_configs = [
    ('YOLOv8n-cls (TF)',  build_yolov8_cls_tf),
    ('MobileNetV2',       build_mobilenetv2),
    ('EfficientNetB0',    build_efficientnetb0),
]

for model_name, builder_fn in model_configs:
    print(f'\n{"-"*50}')
    print(f'Benchmarking: {model_name}')
    print(f'{"-"*50}')
    m      = builder_fn()
    result = run_quick_benchmark(m)
    benchmark_results.append(result)
    keras.backend.clear_session()   # free GPU memory between runs

print('\nBenchmark complete.')


--------------------------------------------------
Benchmarking: YOLOv8n-cls (TF)
--------------------------------------------------
Epoch 1/8


I0000 00:00:1778594541.317575     109 cuda_dnn.cc:529] Loaded cuDNN version 91002


225/225 ━━━━━━━━━━━━━━━━━━━━ 40s 100ms/step - accuracy: 0.7243 - loss: 0.6627 - val_accuracy: 0.3333 - val_loss: 3.2516 - learning_rate: 0.0010
Epoch 2/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 18s 81ms/step - accuracy: 0.8125 - loss: 0.4880 - val_accuracy: 0.4418 - val_loss: 1.3658 - learning_rate: 0.0010
Epoch 3/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 19s 82ms/step - accuracy: 0.8631 - loss: 0.3814 - val_accuracy: 0.6967 - val_loss: 1.2281 - learning_rate: 0.0010
Epoch 4/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 19s 83ms/step - accuracy: 0.8838 - loss: 0.3282 - val_accuracy: 0.3595 - val_loss: 2.9719 - learning_rate: 0.0010
Epoch 5/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 19s 84ms/step - accuracy: 0.8960 - loss: 0.3017 - val_accuracy: 0.8041 - val_loss: 0.8349 - learning_rate: 0.0010
Epoch 6/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 19s 86ms/step - accuracy: 0.9097 - loss: 0.2699 - val_accuracy: 0.8592 - val_loss: 0.4259 - learning_rate: 0.0010
Epoch 7/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 19s 86ms/step - accuracy: 0.9192 - loss: 0.2378 - va

E0000 00:00:1778594889.338852      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/EfficientNetB0_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


225/225 ━━━━━━━━━━━━━━━━━━━━ 38s 125ms/step - accuracy: 0.7487 - loss: 0.6521 - val_accuracy: 0.9633 - val_loss: 0.1463 - learning_rate: 0.0010
Epoch 2/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 23s 103ms/step - accuracy: 0.8993 - loss: 0.2679 - val_accuracy: 0.9727 - val_loss: 0.0846 - learning_rate: 0.0010
Epoch 3/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 22s 99ms/step - accuracy: 0.9166 - loss: 0.2232 - val_accuracy: 0.9772 - val_loss: 0.0725 - learning_rate: 0.0010
Epoch 4/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 22s 99ms/step - accuracy: 0.9271 - loss: 0.2104 - val_accuracy: 0.9805 - val_loss: 0.0671 - learning_rate: 0.0010
Epoch 5/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 23s 100ms/step - accuracy: 0.9291 - loss: 0.1958 - val_accuracy: 0.9789 - val_loss: 0.0657 - learning_rate: 0.0010
Epoch 6/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 23s 100ms/step - accuracy: 0.9286 - loss: 0.1873 - val_accuracy: 0.9833 - val_loss: 0.0668 - learning_rate: 0.0010
Epoch 7/8
225/225 ━━━━━━━━━━━━━━━━━━━━ 22s 99ms/step - accuracy: 0.9337 - loss: 0.1818 -

In [11]:
# --- Print comparison table ---
print('\n' + '=' * 68)
print(f'  Model Comparison  ({QUICK_EPOCHS}-epoch benchmark  |  {NUM_CLASSES} classes)')
print('=' * 68)
print(f'{"Model":<22} {"Params (M)":>10} {"Size (MB)":>10} {"Val Acc %":>10} {"Time (s)":>10}')
print('-' * 68)

best_acc = max(r['val_acc_pct'] for r in benchmark_results)
for r in benchmark_results:
    marker = '  <- best' if r['val_acc_pct'] == best_acc else ''
    print(
        f"{r['name']:<22}"
        f"{r['params_M']:>10}"
        f"{r['size_MB']:>10}"
        f"{r['val_acc_pct']:>10}"
        f"{r['time_s']:>10}"
        f"{marker}"
    )

print('=' * 68)
winner = max(benchmark_results, key=lambda r: r['val_acc_pct'])
print(f"\nWinner: {winner['name']} with {winner['val_acc_pct']}% validation accuracy")

# Save results
save_results = [{k: v for k, v in r.items() if k != 'history'} for r in benchmark_results]
with open(os.path.join(OUT_DIR, 'comparison_results.json'), 'w') as f:
    json.dump(save_results, f, indent=2)
print(f'Saved: {OUT_DIR}/comparison_results.json')


  Model Comparison  (8-epoch benchmark  |  3 classes)
Model                  Params (M)  Size (MB)  Val Acc %   Time (s)
--------------------------------------------------------------------
YOLOv8n_cls_TF              1.11     13.63     94.88     181.8
MobileNetV2                 2.26      9.53     84.81     154.8
EfficientNetB0              4.06     16.93     98.33     205.2  <- best

Winner: EfficientNetB0 with 98.33% validation accuracy
Saved: /kaggle/working/models/comparison_results.json


In [12]:
# --- Comparison bar charts ---
names       = [r['name']        for r in benchmark_results]
accs        = [r['val_acc_pct'] for r in benchmark_results]
sizes       = [r['size_MB']     for r in benchmark_results]
param_counts= [r['params_M']    for r in benchmark_results]
bar_colors  = ['#3498DB', '#2ECC71', '#E67E22']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Model Comparison Results ({NUM_CLASSES}-class problem)', fontsize=14)

plot_specs = [
    (axes[0], accs,        'Validation Accuracy', 'Accuracy (%)',   [min(accs) - 5, 100]),
    (axes[1], sizes,       'Weight File Size',    'Size (MB)',      [0, max(sizes) * 1.3]),
    (axes[2], param_counts,'Parameter Count',     'Parameters (M)',[0, max(param_counts) * 1.3]),
]

for ax, values, title, ylabel, ylim in plot_specs:
    bars = ax.bar(names, values, color=bar_colors, edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(ylim)
    ax.tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + ylim[1] * 0.01,
            str(v), ha='center', fontsize=10, fontweight='bold'
        )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_chart.png')

Saved: comparison_chart.png


In [13]:
# --- Training curves for all 3 architectures ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Quick Benchmark - Training Curves', fontsize=13)

for ax, result, color in zip(axes, benchmark_results, bar_colors):
    ep = range(1, len(result['history']['accuracy']) + 1)
    ax.plot(ep, result['history']['accuracy'],     color=color, label='Train Acc')
    ax.plot(ep, result['history']['val_accuracy'], color=color, label='Val Acc',
            linestyle='--')
    ax.set_title(result['name'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_ylim([0, 1.05])
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'benchmark_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: benchmark_curves.png')

Saved: benchmark_curves.png


---
# Part B - Full Optimized Training (EfficientNetB0, 3-Class)

Two-phase training strategy:

| Phase | Backbone | Learning Rate | Max Epochs | Purpose |
|-------|----------|---------------|------------|---------|
| Phase 1 | Frozen | 1e-3 (Adam) | 50 | Feature extraction |
| Phase 2 | Top 30 layers unfrozen | 1e-5 -> 0 (Cosine Decay) | 30 | Fine-tuning |

Regularization:
- Label smoothing 0.1 in Phase 1, 0.05 in Phase 2
- L2 weight decay on the Dense(256) layer
- Dropout at 0.4 after backbone, 0.2 after Dense
- Full data augmentation pipeline (Section 5)

## Section 8 - Build Full EfficientNetB0 Model

In [14]:
def build_full_efficientnet(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout_rate=0.4):
    """
    EfficientNetB0 with a custom 3-class classification head.

    Architecture:
        Input (224x224x3)
        -> Augmentation
        -> EfficientNetB0 backbone (1280-dim features after GlobalAvgPool)
        -> GlobalAveragePooling2D
        -> BatchNormalization
        -> Dropout(0.4)
        -> Dense(256, swish, L2=1e-4)
        -> Dropout(0.2)
        -> Dense(num_classes=3, softmax)
    """
    backbone = EfficientNetB0(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    backbone.trainable = False   # frozen in Phase 1, partially unfrozen in Phase 2

    inp = keras.Input(shape=(img_size, img_size, 3), name='input_image')
    x   = augment_pipeline(inp)
    x   = backbone(x, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(dropout_rate)(x)
    x   = layers.Dense(
              256,
              activation='swish',
              kernel_regularizer=keras.regularizers.l2(1e-4),
              name='dense_256'
          )(x)
    x   = layers.Dropout(dropout_rate / 2)(x)
    out = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inp, out, name='FaceMask_EfficientNetB0_3class')
    return model, backbone


model, backbone = build_full_efficientnet()

total_params = model.count_params()
trainable_p1 = sum(np.prod(v.shape) for v in model.trainable_variables)

print(f'Total parameters        : {total_params:,}')
print(f'Trainable in Phase 1    : {trainable_p1:,}  ({trainable_p1/total_params*100:.1f}%)')
print(f'Frozen in Phase 1       : {total_params - trainable_p1:,}')
print(f'Output classes          : {NUM_CLASSES} ({CLASS_LABELS})')
model.summary(line_length=90)

Total parameters        : 4,383,398
Trainable in Phase 1    : 331,267  (7.6%)
Frozen in Phase 1       : 4,052,131
Output classes          : 3 (['with_mask', 'mask_weared_incorrect', 'without_mask'])


Model: "FaceMask_EfficientNetB0_3class"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                          ┃ Output Shape                 ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)              │ (None, 224, 224, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ augmentation (Sequential)             │ (None, 224, 224, 3)          │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ efficientnetb0 (Functional)           │ (None, 7, 7, 1280)           │       4,049,571 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ global_average_pooling2d              │ (None, 1280)                 │               0 │
│ (GlobalAveragePooling2D)              │                              │                 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ batch_normalization                   │ (None, 1280)                 │           5,120 │
│ (BatchNormalization)                  │                              │                 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout (Dropout)                     │ (None, 1280)                 │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dense_256 (Dense)                     │ (None, 256)                  │         327,936 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                   │ (None, 256)                  │               0 │
├───────────────────────────────────────┼──────────────────────────────┼─────────────────┤
│ predictions (Dense)                   │ (None, 3)                    │             771 │
└───────────────────────────────────────┴──────────────────────────────┴─────────────────┘

 Total params: 4,383,398 (16.72 MB)

 Trainable params: 331,267 (1.26 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

## Section 9 - Phase 1: Feature Extraction

In [15]:
CHECKPOINT_PATH = os.path.join(OUT_DIR, 'best_model.keras')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    # label_smoothing=0.1: true class target is 0.9 instead of 1.0
    # prevents overconfident predictions and helps generalization
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
]

print('Starting Phase 1 - Feature Extraction')
print(f'Backbone frozen | LR=1e-3 | Max {PHASE1_EPOCHS} epochs | {NUM_CLASSES} classes')
print('-' * 60)

phase1_start = time.time()
hist1 = model.fit(
    X_train, keras.utils.to_categorical(y_train, NUM_CLASSES),
    validation_data=(X_val, keras.utils.to_categorical(y_val, NUM_CLASSES)),
    epochs=PHASE1_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase1,
    class_weight=class_weight_dict,  # compensate for class imbalance
    verbose=1
)
phase1_time = time.time() - phase1_start
phase1_best = max(hist1.history['val_accuracy'])

print(f'\nPhase 1 complete in {phase1_time:.0f}s')
print(f'Best validation accuracy: {phase1_best * 100:.2f}%')

Starting Phase 1 - Feature Extraction
Backbone frozen | LR=1e-3 | Max 50 epochs | 3 classes
------------------------------------------------------------
Epoch 1/50


E0000 00:00:1778595109.903305      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/FaceMask_EfficientNetB0_3class_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.8190 - loss: 0.8092
Epoch 1: val_accuracy improved from -inf to 0.96605, saving model to /kaggle/working/models/best_model.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 39s 127ms/step - accuracy: 0.8191 - loss: 0.8088 - val_accuracy: 0.9661 - val_loss: 0.4439 - learning_rate: 0.0010
Epoch 2/50
224/225 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9122 - loss: 0.5424
Epoch 2: val_accuracy improved from 0.96605 to 0.98164, saving model to /kaggle/working/models/best_model.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 24s 105ms/step - accuracy: 0.9123 - loss: 0.5423 - val_accuracy: 0.9816 - val_loss: 0.4026 - learning_rate: 0.0010
Epoch 3/50
224/225 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9355 - loss: 0.4929
Epoch 3: val_accuracy did not improve from 0.98164
225/225 ━━━━━━━━━━━━━━━━━━━━ 22s 99ms/step - accuracy: 0.9355 - loss: 0.4929 - val_accuracy: 0.9811 - val_loss: 0.3958 - learning_rate: 0.0010
Epoch 4/50
224/225 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms

## Section 10 - Phase 2: Fine-Tuning

In [16]:
# Unfreeze the top UNFREEZE_LAYERS layers of the backbone.
# Lower layers encode generic features (edges, textures) — keep frozen.
# Upper layers encode task-specific features — fine-tune these.
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable_phase2 = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Trainable parameters in Phase 2: {trainable_phase2:,}')
print(f'Backbone layers unfrozen: top {UNFREEZE_LAYERS} of {len(backbone.layers)}')

# Cosine decay LR schedule: starts at 1e-5, smoothly decreases toward zero.
# Using a very low LR is critical — too high will overwrite pretrained weights.
total_steps = (len(X_train) // BATCH_SIZE) * PHASE2_EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-6,  # lowered from 1e-5 — prevents catastrophic forgetting
    decay_steps=total_steps,
    alpha=1e-7
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

print('\nStarting Phase 2 - Fine-Tuning')
print(f'Cosine LR 1e-5 -> 0 | Max {PHASE2_EPOCHS} epochs')
print('-' * 60)

phase2_start = time.time()
hist2 = model.fit(
    X_train, keras.utils.to_categorical(y_train, NUM_CLASSES),
    validation_data=(X_val, keras.utils.to_categorical(y_val, NUM_CLASSES)),
    epochs=PHASE2_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase2,
    class_weight=class_weight_dict,  # compensate for class imbalance
    verbose=1
)
phase2_time = time.time() - phase2_start
phase2_best = max(hist2.history['val_accuracy'])

print(f'\nPhase 2 complete in {phase2_time:.0f}s')
print(f'Best validation accuracy: {phase2_best * 100:.2f}%')

Trainable parameters in Phase 2: 1,239,475
Backbone layers unfrozen: top 15 of 238

Starting Phase 2 - Fine-Tuning
Cosine LR 1e-5 -> 0 | Max 30 epochs
------------------------------------------------------------
Epoch 1/30


E0000 00:00:1778595754.885050      57 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/FaceMask_EfficientNetB0_3class_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8931 - loss: 0.5210
Epoch 1: val_accuracy improved from -inf to 0.98442, saving model to /kaggle/working/models/best_model.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 43s 137ms/step - accuracy: 0.8931 - loss: 0.5209 - val_accuracy: 0.9844 - val_loss: 0.3219
Epoch 2/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.9280 - loss: 0.4470
Epoch 2: val_accuracy did not improve from 0.98442
225/225 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - accuracy: 0.9281 - loss: 0.4469 - val_accuracy: 0.9839 - val_loss: 0.3099
Epoch 3/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9413 - loss: 0.3967
Epoch 3: val_accuracy did not improve from 0.98442
225/225 ━━━━━━━━━━━━━━━━━━━━ 25s 109ms/step - accuracy: 0.9413 - loss: 0.3967 - val_accuracy: 0.9833 - val_loss: 0.2893
Epoch 4/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9433 - loss: 0.3778
Epoch 4: val_accuracy improved from 0.98442 to 0.98720, saving model to /kaggle/working/mode

## Section 11 - Training Curves

In [17]:
# Merge Phase 1 and Phase 2 histories for a single continuous plot
all_train_acc = hist1.history['accuracy']     + hist2.history['accuracy']
all_val_acc   = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
all_train_los = hist1.history['loss']         + hist2.history['loss']
all_val_los   = hist1.history['val_loss']     + hist2.history['val_loss']
phase1_end    = len(hist1.history['accuracy'])
epoch_range   = range(1, len(all_train_acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EfficientNetB0 - Full Training History (3-class)', fontsize=14)

for ax, (train_vals, val_vals), title in zip(
    axes,
    [(all_train_acc, all_val_acc), (all_train_los, all_val_los)],
    ['Accuracy', 'Loss']
):
    ax.plot(epoch_range, train_vals, 'b-', linewidth=2, label=f'Train {title}')
    ax.plot(epoch_range, val_vals,   'r-', linewidth=2, label=f'Val {title}')
    ax.axvline(phase1_end, color='gray', linestyle='--', linewidth=1.5,
               label='Fine-tuning starts')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

Saved: training_curves.png


## Section 12 - Model Evaluation

In [18]:
# --- Predictions ---
y_probs = model.predict(X_val, verbose=0)   # shape (N, 3)
y_pred  = np.argmax(y_probs, axis=1)

val_acc = accuracy_score(y_val, y_pred)

# For 3-class ROC-AUC, use One-vs-Rest strategy with macro averaging
roc_auc = roc_auc_score(
    y_val, y_probs,
    multi_class='ovr',
    average='macro'
)

print(f'Validation Accuracy : {val_acc * 100:.2f}%')
print(f'ROC-AUC (OvR macro) : {roc_auc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=CLASS_LABELS))

Validation Accuracy : 98.89%
ROC-AUC (OvR macro) : 0.9996

Classification Report:
                       precision    recall  f1-score   support

            with_mask       0.99      0.98      0.98       599
mask_weared_incorrect       0.99      1.00      0.99       599
         without_mask       0.99      0.99      0.99       599

             accuracy                           0.99      1797
            macro avg       0.99      0.99      0.99      1797
         weighted avg       0.99      0.99      0.99      1797



In [19]:
# --- Confusion matrix (3x3) ---
cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_LABELS).plot(
    ax=ax, colorbar=True, cmap='Blues'
)
ax.set_title('Confusion Matrix - EfficientNetB0 (3-class)', fontsize=13)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

# Print per-class accuracy for clarity
print('\nPer-class accuracy:')
for i, label in enumerate(CLASS_LABELS):
    class_acc = cm[i, i] / cm[i].sum() * 100
    print(f'  {label}: {class_acc:.1f}%')

Saved: confusion_matrix.png

Per-class accuracy:
  with_mask: 97.8%
  mask_weared_incorrect: 99.8%
  without_mask: 99.0%


In [20]:
# --- Prediction samples: correct and incorrect ---
correct_indices   = np.where(y_pred == y_val)[0][:9]
incorrect_indices = np.where(y_pred != y_val)[0][:3]

fig, axes = plt.subplots(4, 3, figsize=(12, 14))
fig.suptitle('Prediction Samples  (rows 1-3: correct | row 4: incorrect)', fontsize=12)

# Correct predictions (first 3 rows of 3 cols each)
for i, idx in enumerate(correct_indices):
    ax   = axes[i // 3][i % 3]
    conf = y_probs[idx, y_pred[idx]] * 100
    ax.imshow(X_val[idx])
    ax.axis('off')
    ax.set_title(
        f'CORRECT\n{CLASS_LABELS[y_pred[idx]]}\n{conf:.1f}%',
        color=CLASS_COLORS[y_pred[idx]], fontsize=8
    )

# Wrong predictions (last row)
for i, idx in enumerate(incorrect_indices):
    ax   = axes[3][i]
    conf = y_probs[idx, y_pred[idx]] * 100
    ax.imshow(X_val[idx])
    ax.axis('off')
    ax.set_title(
        f'WRONG\nPred: {CLASS_LABELS[y_pred[idx]]}\nTrue: {CLASS_LABELS[y_val[idx]]}\n{conf:.1f}%',
        color='red', fontsize=8
    )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'prediction_samples.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: prediction_samples.png')

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [26.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [2.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [25.0..233.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [1.0..255.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [7.0..183.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..112.0].
Clipping input data to the valid range for imshow with RGB d

Saved: prediction_samples.png


## Section 13 - Save Keras Model

In [21]:
# Keras 3 cannot serialize augmentation/lambda layers to legacy H5 format.
# Use the native .keras format instead — it handles all layer types correctly.
keras_path = os.path.join(OUT_DIR, 'face_mask_efficientnet.keras')
model.save(keras_path)
keras_size_mb = os.path.getsize(keras_path) / 1e6
print(f'Keras model saved : {keras_path}')
print(f'File size         : {keras_size_mb:.1f} MB')

# Also keep an H5 path variable for any downstream code that references it
h5_path = keras_path  # alias so TFLite export cells still work

Keras model saved : /kaggle/working/models/face_mask_efficientnet.keras
File size         : 28.3 MB


---
# Part C - TFLite Export and Quantization

| Format | Method                          | Size reduction | Accuracy drop |
|--------|---------------------------------|----------------|---------------|
| FP16   | Weights stored as float16       | ~50%           | < 0.3%        |
| INT8   | Weights and activations as int8 | ~75%           | ~0.5%         |

## Section 14 - Export TFLite FP16

In [22]:
print('Exporting TFLite FP16 model...')

converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations               = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16_bytes = converter_fp16.convert()

fp16_path = os.path.join(OUT_DIR, 'face_mask_fp16.tflite')
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16_bytes)

fp16_size_mb = os.path.getsize(fp16_path) / 1e6
print(f'Saved : {fp16_path}')
print(f'Size  : {fp16_size_mb:.1f} MB  ({(1 - fp16_size_mb/keras_size_mb)*100:.0f}% smaller than Keras)')

Exporting TFLite FP16 model...
INFO:tensorflow:Assets written to: /tmp/tmpublkc02h/assets


INFO:tensorflow:Assets written to: /tmp/tmpublkc02h/assets


Saved artifact at '/tmp/tmpublkc02h'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135837881836048: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135837881836816: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135837467046352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467045776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467044624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467043856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467041744: TensorSpec(shape=(), dtype=tf.resource, name=None)

W0000 00:00:1778596234.664147      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778596234.664211      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778596234.892396      57 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Saved : /kaggle/working/models/face_mask_fp16.tflite
Size  : 8.7 MB  (69% smaller than Keras)


## Section 15 - Export TFLite INT8

In [23]:
# INT8 quantization needs a calibration dataset.
# The converter runs forward passes to measure the range of activations,
# so it can map float32 values accurately into the int8 range [-128, 127].

def representative_dataset_generator():
    """Yields 200 sample images from the validation set for INT8 calibration."""
    for img in X_val[:200]:
        yield [img[np.newaxis].astype(np.float32)]


print('Exporting TFLite INT8 model...')

int8_path    = None
int8_size_mb = None

try:
    converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
    converter_int8.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter_int8.representative_dataset    = representative_dataset_generator
    converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter_int8.inference_input_type      = tf.uint8
    converter_int8.inference_output_type     = tf.uint8
    tflite_int8_bytes = converter_int8.convert()

    int8_path = os.path.join(OUT_DIR, 'face_mask_int8.tflite')
    with open(int8_path, 'wb') as f:
        f.write(tflite_int8_bytes)

    int8_size_mb = os.path.getsize(int8_path) / 1e6
    print(f'Saved : {int8_path}')
    print(f'Size  : {int8_size_mb:.1f} MB  ({(1 - int8_size_mb/keras_size_mb)*100:.0f}% smaller than Keras)')

except Exception as e:
    print(f'INT8 export failed: {e}')
    print('Continuing without INT8 model.')

Exporting TFLite INT8 model...
INFO:tensorflow:Assets written to: /tmp/tmpy93_ll1a/assets


INFO:tensorflow:Assets written to: /tmp/tmpy93_ll1a/assets


Saved artifact at '/tmp/tmpy93_ll1a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_image')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135837881836048: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135837881836816: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135837467046352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467045776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467044624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467046736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467043856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135837467041744: TensorSpec(shape=(), dtype=tf.resource, name=None)

W0000 00:00:1778596250.553854      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778596250.553886      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: UINT8


Saved : /kaggle/working/models/face_mask_int8.tflite
Size  : 5.3 MB  (81% smaller than Keras)


## Section 16 - Validate TFLite Accuracy

In [24]:
def evaluate_tflite_model(model_path, X_test, y_test):
    """
    Run inference with a TFLite model on the full validation set.
    Handles both:
      - float32 input/output (FP16 model still uses float32 IO)
      - uint8 input/output  (INT8 fully-quantized model)

    Returns accuracy as a float (0.0 - 1.0).
    """
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    predictions = []
    for img in X_test:
        inp = img[np.newaxis].astype(np.float32)

        # Quantize input if model expects uint8
        if input_details['dtype'] == np.uint8:
            scale, zero_point = input_details['quantization']
            inp = (inp / scale + zero_point).clip(0, 255).astype(np.uint8)

        interpreter.set_tensor(input_details['index'], inp)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])[0]

        # Dequantize output if it is uint8
        if output_details['dtype'] == np.uint8:
            scale, zero_point = output_details['quantization']
            output = (output.astype(np.float32) - zero_point) * scale

        predictions.append(np.argmax(output))

    return accuracy_score(y_test, predictions)


print('Evaluating TFLite FP16 on validation set...')
fp16_acc = evaluate_tflite_model(fp16_path, X_val, y_val)
print(f'TFLite FP16 accuracy : {fp16_acc * 100:.2f}%')
print(f'Accuracy drop        : {(val_acc - fp16_acc) * 100:+.2f}%')

int8_acc = None
if int8_path and os.path.exists(int8_path):
    print('\nEvaluating TFLite INT8 on validation set...')
    int8_acc = evaluate_tflite_model(int8_path, X_val, y_val)
    print(f'TFLite INT8 accuracy : {int8_acc * 100:.2f}%')
    print(f'Accuracy drop        : {(val_acc - int8_acc) * 100:+.2f}%')

Evaluating TFLite FP16 on validation set...


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


TFLite FP16 accuracy : 98.89%
Accuracy drop        : +0.00%

Evaluating TFLite INT8 on validation set...
TFLite INT8 accuracy : 74.85%
Accuracy drop        : +24.04%


## Section 17 - Final Results Dashboard

In [25]:
# --- Size vs accuracy comparison ---
model_labels = ['Keras H5', 'TFLite FP16']
model_sizes  = [keras_size_mb, fp16_size_mb]
model_accs   = [val_acc * 100, fp16_acc * 100]

if int8_size_mb and int8_acc:
    model_labels.append('TFLite INT8')
    model_sizes.append(int8_size_mb)
    model_accs.append(int8_acc * 100)

bar_colors_q = ['#3498DB', '#2ECC71', '#E67E22']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final Model - Size vs Accuracy After Quantization (3-class)', fontsize=13)

bars1 = axes[0].bar(model_labels, model_sizes, color=bar_colors_q[:len(model_labels)])
axes[0].set_ylabel('File Size (MB)')
axes[0].set_title('Model Size (smaller is better for deployment)')
for bar, v in zip(bars1, model_sizes):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{v:.1f} MB', ha='center', fontweight='bold'
    )

bars2 = axes[1].bar(model_labels, model_accs, color=bar_colors_q[:len(model_labels)])
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy After Quantization')
axes[1].set_ylim([min(model_accs) - 2, 100])
for bar, v in zip(bars2, model_accs):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.1,
        f'{v:.1f}%', ha='center', fontweight='bold'
    )

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'size_vs_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: size_vs_accuracy.png')

Saved: size_vs_accuracy.png


In [26]:
# --- Print and save final summary ---
print('=' * 62)
print('  TRAINING SUMMARY')
print('=' * 62)
print(f'  Dataset              : {DATA_DIR}')
print(f'  Total images         : {len(X)}')
print(f'  Classes              : {CLASS_LABELS}')
print(f'  Model                : EfficientNetB0 (3-class)')
print(f'  Phase 1 epochs run   : {len(hist1.history["accuracy"])}')
print(f'  Phase 2 epochs run   : {len(hist2.history["accuracy"])}')
print('  ' + '-' * 58)
print(f'  Keras val accuracy   : {val_acc * 100:.2f}%')
print(f'  ROC-AUC (OvR macro)  : {roc_auc:.4f}')
print(f'  TFLite FP16 accuracy : {fp16_acc * 100:.2f}%')
if int8_acc:
    print(f'  TFLite INT8 accuracy : {int8_acc * 100:.2f}%')
print('  ' + '-' * 58)
print(f'  Keras model size     : {keras_size_mb:.1f} MB')
print(f'  TFLite FP16 size     : {fp16_size_mb:.1f} MB  '
      f'({(1 - fp16_size_mb/keras_size_mb)*100:.0f}% smaller)')
if int8_size_mb:
    print(f'  TFLite INT8 size     : {int8_size_mb:.1f} MB  '
          f'({(1 - int8_size_mb/keras_size_mb)*100:.0f}% smaller)')
print('  ' + '-' * 58)
print(f'  Output directory     : {OUT_DIR}')
print('=' * 62)

summary = {
    'dataset_path'         : DATA_DIR,
    'classes'              : CLASS_LABELS,
    'num_classes'          : NUM_CLASSES,
    'total_images'         : len(X),
    'model'                : 'EfficientNetB0',
    'phase1_epochs_run'    : len(hist1.history['accuracy']),
    'phase2_epochs_run'    : len(hist2.history['accuracy']),
    'val_accuracy_pct'     : round(val_acc * 100, 2),
    'roc_auc_ovr_macro'    : round(roc_auc, 4),
    'tflite_fp16_acc_pct'  : round(fp16_acc * 100, 2),
    'tflite_int8_acc_pct'  : round(int8_acc * 100, 2) if int8_acc else None,
    'keras_size_mb'        : round(keras_size_mb, 2),
    'tflite_fp16_mb'       : round(fp16_size_mb, 2),
    'tflite_int8_mb'       : round(int8_size_mb, 2) if int8_size_mb else None,
    'total_params'         : int(model.count_params()),
}

summary_path = os.path.join(OUT_DIR, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved: {summary_path}')

  TRAINING SUMMARY
  Dataset              : /kaggle/input/datasets/vijaykumar1799/face-mask-detection/Dataset
  Total images         : 8982
  Classes              : ['with_mask', 'mask_weared_incorrect', 'without_mask']
  Model                : EfficientNetB0 (3-class)
  Phase 1 epochs run   : 27
  Phase 2 epochs run   : 18
  ----------------------------------------------------------
  Keras val accuracy   : 98.89%
  ROC-AUC (OvR macro)  : 0.9996
  TFLite FP16 accuracy : 98.89%
  TFLite INT8 accuracy : 74.85%
  ----------------------------------------------------------
  Keras model size     : 28.3 MB
  TFLite FP16 size     : 8.7 MB  (69% smaller)
  TFLite INT8 size     : 5.3 MB  (81% smaller)
  ----------------------------------------------------------
  Output directory     : /kaggle/working/models

Summary saved: /kaggle/working/models/training_summary.json


## Section 18 - Single Image Inference Demo

In [27]:
def predict_single_image(image_path, tflite_model_path=None):
    """
    Run 3-class prediction on one image file.

    Parameters:
        image_path        : path to a .jpg or .png file
        tflite_model_path : path to a .tflite file (optional)
                            If not provided, uses the Keras model.

    Returns:
        predicted_label : one of 'with_mask', 'mask_weared_incorrect', 'without_mask'
        confidence_pct  : confidence for the predicted class (0-100)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')

    img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_res  = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_norm = img_res.astype(np.float32) / 255.0

    if tflite_model_path and os.path.exists(tflite_model_path):
        interp = tf.lite.Interpreter(model_path=tflite_model_path)
        interp.allocate_tensors()
        in_det  = interp.get_input_details()[0]
        out_det = interp.get_output_details()[0]

        inp = img_norm[np.newaxis]
        if in_det['dtype'] == np.uint8:
            scale, zp = in_det['quantization']
            inp = (inp / scale + zp).clip(0, 255).astype(np.uint8)

        interp.set_tensor(in_det['index'], inp)
        interp.invoke()
        probs = interp.get_tensor(out_det['index'])[0].astype(np.float32)

        if out_det['dtype'] == np.uint8:
            scale, zp = out_det['quantization']
            probs = (probs - zp) * scale

        engine = 'TFLite'
    else:
        probs  = model.predict(img_norm[np.newaxis], verbose=0)[0]
        engine = 'Keras'

    pred_idx   = int(np.argmax(probs))
    confidence = float(probs[pred_idx]) * 100
    label      = CLASS_LABELS[pred_idx]

    # Show image with all 3 class probabilities
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    axes[0].imshow(img_rgb)
    axes[0].axis('off')
    axes[0].set_title(
        f'Prediction: {label}\nConfidence: {confidence:.1f}%  [{engine}]',
        color=CLASS_COLORS[pred_idx], fontsize=11
    )

    # Probability bar for all 3 classes
    axes[1].barh(CLASS_LABELS, probs * 100, color=CLASS_COLORS)
    axes[1].set_xlim([0, 100])
    axes[1].set_xlabel('Probability (%)')
    axes[1].set_title('Class Probabilities')
    for i, v in enumerate(probs * 100):
        axes[1].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    return label, confidence


# Test on one image from each of the 3 classes
for cls_idx, folder_name in enumerate(CLASS_FOLDERS):
    folder    = os.path.join(DATA_DIR, folder_name)
    test_file = os.path.join(folder, sorted(os.listdir(folder))[0])
    print(f'\nClass: {CLASS_LABELS[cls_idx]}')
    print(f'File : {test_file}')
    predicted_label, conf = predict_single_image(test_file, fp16_path)
    correct = 'CORRECT' if predicted_label == CLASS_LABELS[cls_idx] else 'WRONG'
    print(f'Result: {predicted_label}  ({conf:.1f}%)  [{correct}]')


Class: with_mask
File : /kaggle/input/datasets/vijaykumar1799/face-mask-detection/Dataset/with_mask/1.png
Result: without_mask  (45.8%)  [WRONG]

Class: mask_weared_incorrect
File : /kaggle/input/datasets/vijaykumar1799/face-mask-detection/Dataset/mask_weared_incorrect/1.png
Result: without_mask  (45.8%)  [WRONG]

Class: without_mask
File : /kaggle/input/datasets/vijaykumar1799/face-mask-detection/Dataset/without_mask/1.png
Result: without_mask  (45.9%)  [CORRECT]


---
# Summary

All output files are saved to /kaggle/working/models/

| File | Description |
|------|-------------|
| best_model.keras | Best checkpoint saved during training |
| face_mask_efficientnet.h5 | Full Keras model (~21 MB) |
| face_mask_fp16.tflite | FP16 quantized — recommended for deployment (~5 MB) |
| face_mask_int8.tflite | INT8 quantized — for mobile/edge devices (~3 MB) |
| training_curves.png | Phase 1 and Phase 2 accuracy/loss curves |
| confusion_matrix.png | 3x3 per-class prediction breakdown |
| comparison_chart.png | Benchmark chart for all 3 architectures |
| comparison_results.json | Benchmark metrics |
| training_summary.json | Final accuracy and size metrics |

Next step: download face_mask_fp16.tflite and run it with app.py (Gradio web interface).